In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time


In [2]:
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/_NIFTY_IDX__202507041318.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

# Save and reload to Parquet (optional)
try:

    print("spot_df loaded successfully from parquet.")
except Exception as e:
    print(f"Error saving/loading spot_df: {e}")


spot_df preview:
             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20
spot_df loaded successfully from parquet.


In [3]:
# load spot data

# --- 2. Ensure Datetime is datetime and sort for time-based ops ---
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# --- 3. Set Datetime as index for resampling ---
spot_df = spot_df.set_index('Datetime')

# --- 4. Filter to regular NIFTY trading hours (avoid pre/post-market ticks) ---
spot_df = spot_df.between_time('09:15:00', '15:15:00')

# --- 5. Resample to 1-min OHLC bars ---
spot_1min = spot_df.resample(
    '1min',
    origin='start_day',
    label='left',
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_1min = spot_1min.reset_index()

# --- 7. Calculate 200-period EMA on Close for trend filter ---
spot_1min['EMA_200'] = spot_1min['Close'].ewm(span=200, adjust=False).mean()

# --- 9. Preview final 1-min OHLC + EMA DataFrame ---
print(spot_1min.tail())


                  Datetime      Open      High       Low     Close  \
359730 2025-06-13 15:11:00  24719.70  24722.95  24718.35  24719.75   
359731 2025-06-13 15:12:00  24719.10  24720.85  24717.20  24720.00   
359732 2025-06-13 15:13:00  24719.35  24725.05  24719.35  24723.50   
359733 2025-06-13 15:14:00  24723.30  24728.45  24723.30  24725.60   
359734 2025-06-13 15:15:00  24725.15  24725.35  24709.75  24711.70   

             EMA_200  
359730  24704.990851  
359731  24705.140196  
359732  24705.322881  
359733  24705.524643  
359734  24705.586089  


In [ ]:
import os, glob, pickle
import pandas as pd

# ================================
# LOAD OPTION DATA YEARWISE (KEEP DATE/TIME SEPARATE, NO SAVE)
# ================================

years_to_load = [2021, 2022, 2023, 2024, 2025]

all_data = {}  # dict to hold DataFrames year-wise

for year in years_to_load:
    print(f"\n--- Loading option data for year: {year} ---")

    options_files = glob.glob(f"/home/newberry3/main/Data/NIFTY/NIFTY_{year}*.pkl")
    if not options_files:
        print(f"No .pkl files found for {year}")
        continue

    dfs = []
    for file in options_files:
        print(f"  Reading {file} ...")
        df = pickle.load(open(file, "rb"))

        # Drop unused columns if present
        cols_to_drop = ['OI', 'Volume', 'Ticker', 'High', 'Low', 'Close']
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

        # Ensure types
        if 'StrikePrice' in df.columns:
            df['StrikePrice'] = df['StrikePrice'].astype(float)
        if 'Type' in df.columns:
            df['Type'] = df['Type'].astype(str).str.strip().str.upper()
        if 'ExpiryDate' in df.columns:
            df['ExpiryDate'] = pd.to_datetime(df['ExpiryDate'], errors='coerce').dt.normalize()
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.date.astype(str)
        if 'Time' in df.columns:
            df['Time'] = pd.to_datetime(df['Time'], errors='coerce').dt.time.astype(str)

        # Reorder: Date, Time first
        cols = ['Date', 'Time'] + [c for c in df.columns if c not in ['Date', 'Time']]
        df = df[cols]

        dfs.append(df)

    # Combine all pickles for the year
    year_df = pd.concat(dfs, ignore_index=True)
    all_data[year] = year_df
    print(f"✅ Loaded {year}, shape={year_df.shape}")

print("\nAll option data loaded into memory.")



--- Loading option data for year: 2021 ---
  Reading /home/newberry3/main/Data/NIFTY/NIFTY_202110.pkl ...


/tmp/ipykernel_2304/3825046238.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time'], errors='coerce').dt.time.astype(str)


  Reading /home/newberry3/main/Data/NIFTY/NIFTY_202109.pkl ...


/tmp/ipykernel_2304/3825046238.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time'], errors='coerce').dt.time.astype(str)


  Reading /home/newberry3/main/Data/NIFTY/NIFTY_202106.pkl ...


...

  Reading /home/newberry3/main/Data/NIFTY/NIFTY_202505.pkl ...


/tmp/ipykernel_2304/3825046238.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time'], errors='coerce').dt.time.astype(str)


  Reading /home/newberry3/main/Data/NIFTY/NIFTY_202504.pkl ...


/tmp/ipykernel_2304/3825046238.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time'], errors='coerce').dt.time.astype(str)


✅ Loaded 2025, shape=(5656160, 6)

All option data loaded into memory.


In [5]:
df_all= all_data
df_all

{2021:                Date      Time ExpiryDate  StrikePrice Type  Open
 0        2021-10-01  09:21:00 2021-10-07      14800.0   PE  2.00
 1        2021-10-01  09:22:00 2021-10-07      14800.0   PE  1.80
 2        2021-10-01  09:23:00 2021-10-07      14800.0   PE  1.65
 3        2021-10-01  09:26:00 2021-10-07      14800.0   PE  1.50
 4        2021-10-01  09:27:00 2021-10-07      14800.0   PE  1.35
 ...             ...       ...        ...          ...  ...   ...
 3994006  2021-07-30  15:26:00 2021-08-05      17400.0   CE  0.75
 3994007  2021-07-30  15:27:00 2021-08-05      17400.0   CE  0.75
 3994008  2021-07-30  15:28:00 2021-08-05      17400.0   CE  0.70
 3994009  2021-07-30  15:29:00 2021-08-05      17400.0   CE  0.75
 3994010  2021-07-30  15:30:00 2021-08-05      17400.0   CE  0.75
 
 [3994011 rows x 6 columns],
 2022:                Date      Time ExpiryDate  StrikePrice Type    Open
 0        2022-09-01  09:15:00 2022-09-01      18750.0   CE    0.45
 1        2022-09-01  09:16:0

In [6]:
def get_strike(ATM, minute, daily_option_data):
    
    def get_premium(option_type):
        
        subset = temp_option_data[(temp_option_data['StrikePrice'] == ATM) & (temp_option_data['Type'] == option_type)]
        
        if subset.empty:
            return None
        
        return subset['Open'].iloc[0]
    
    def find_nearest_strike(target_premium, option_type):
        
        subset = temp_option_data[temp_option_data['Type'] == option_type]
        
        subset = subset.reset_index(drop=True)
        if subset.empty:
            return None, None
        
        nearest_index = (subset['Open'] - target_premium).abs().idxmin()
        nearest_strike = subset.loc[nearest_index, 'StrikePrice']
        nearest_premium = subset.loc[nearest_index, 'Open']
        
        return nearest_strike, nearest_premium
    
    time = minute.strftime('%H:%M:%S')

    # Convert the filter time to a time object
    filter_time_object = pd.to_datetime(time).time()

    # Filter the DataFrame for the specific time
    temp_option_data = daily_option_data[daily_option_data.index.time == filter_time_object]
    #temp_option_data = daily_option_data[daily_option_data['Time'] == time]
    
    CE_ATM_premium = get_premium('CE')
    PE_ATM_premium = get_premium('PE')
    
    if (CE_ATM_premium is None) or (PE_ATM_premium is None):
        return None, None, None, None, None, None
    
    CE_OTM_premium_expected = CE_ATM_premium * STRIKE
    PE_OTM_premium_expected = PE_ATM_premium * STRIKE
    
    
    CE_OTM, CE_OTM_premium = find_nearest_strike(CE_OTM_premium_expected, 'CE')
    PE_OTM, PE_OTM_premium = find_nearest_strike(PE_OTM_premium_expected, 'PE')
    
    return CE_ATM_premium, PE_ATM_premium, CE_OTM, CE_OTM_premium, PE_OTM, PE_OTM_premium

In [7]:
import pandas as pd
from datetime import datetime

# ====== CONFIG YOU CAN TWEAK ======
TARGET_DATE_STR = "20-10-2023"     # dd-mm-yyyy
TIMES_TO_CHECK  = ["10:30:00", "12:00:00", "14:00:00"]  # HH:MM:SS
STRIKE          = 0.40             # your premium multiplier used by get_strike()
INSTRUMENT      = "NIFTY"          # or "SENSEX"
STEP_BY_INST    = {"NIFTY": 50, "SENSEX": 100}  # ATM step
ATM_SOURCE_COL  = "Close"          # use Close from spot_1min to compute ATM

# ====== HELPERS ======
def round_to_step(x, step):
    return int(round(x / step) * step)

def build_options_day(all_data: dict, target_date_str: str) -> pd.DataFrame:
    """Concatenate all years, reconstruct DateTime, slice one day, return minute-indexed options DF."""
    # concat all years already loaded into memory
    options_all = pd.concat(all_data.values(), ignore_index=True)

    # reconstruct DateTime from kept Date/Time strings
    # (your earlier pipeline already cleaned them)
    options_all["DateTime"] = pd.to_datetime(
        options_all["Date"].astype(str) + " " + options_all["Time"].astype(str),
        errors="coerce"
    )
    options_all = options_all.dropna(subset=["DateTime"])

    # filter to target date
    target_date = pd.to_datetime(target_date_str, format="%d-%m-%Y").date()
    options_day = options_all[options_all["DateTime"].dt.date == target_date].copy()

    # set minute index
    options_day = options_day.set_index("DateTime").sort_index()

    # standardize essential cols just in case
    options_day["Type"] = options_day["Type"].astype(str).str.upper().str.strip()
    options_day["StrikePrice"] = options_day["StrikePrice"].astype(float)

    return options_day

def get_atm_from_spot(spot_1min: pd.DataFrame, minute_ts: pd.Timestamp, instrument: str) -> int:
    """Pick ATM by rounding the spot Close to the exchange step for that instrument at the given minute."""
    # ensure spot_1min has Datetime
    s = spot_1min.set_index("Datetime").sort_index()
    if minute_ts not in s.index:
        # try nearest minute within same day (optional fallback)
        nearest = s.index.get_indexer([minute_ts], method="nearest")
        if nearest[0] == -1:
            raise ValueError(f"No spot data for {minute_ts}")
        minute_ts = s.index[nearest[0]]

    spot_val = float(s.loc[minute_ts, ATM_SOURCE_COL])
    step = STEP_BY_INST.get(instrument.upper(), 50)
    return round_to_step(spot_val, step)

# ====== BUILD daily_option_data FOR THE TARGET DATE ======
daily_option_data = build_options_day(all_data, TARGET_DATE_STR)
if daily_option_data.empty:
    raise ValueError(f"No options data found for {TARGET_DATE_STR}")

# Ensure the index is DatetimeIndex (required by your get_strike)
assert isinstance(daily_option_data.index, pd.DatetimeIndex)

# ====== RUN CHECKS FOR SPECIFIC MINUTES ======
results = []
for t in TIMES_TO_CHECK:
    minute_ts = pd.Timestamp(datetime.strptime(f"{TARGET_DATE_STR} {t}", "%d-%m-%Y %H:%M:%S"))

    # Compute ATM from spot_1min for that minute
    try:
        ATM = get_atm_from_spot(spot_1min, minute_ts, INSTRUMENT)
    except Exception as e:
        print(f"[{minute_ts}] ATM error: {e}")
        continue

    # Call your function (which expects: ATM, minute_ts, daily_option_data)
    out = get_strike(ATM, minute_ts, daily_option_data)

    # Unpack and log neatly
    (CE_ATM_prem, PE_ATM_prem, CE_OTM, CE_OTM_prem, PE_OTM, PE_OTM_prem) = out
    results.append({
        "Minute": minute_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "ATM": ATM,
        "CE_ATM_premium": CE_ATM_prem,
        "PE_ATM_premium": PE_ATM_prem,
        "CE_OTM_strike": CE_OTM,
        "CE_OTM_premium": CE_OTM_prem,
        "PE_OTM_strike": PE_OTM,
        "PE_OTM_premium": PE_OTM_prem
    })

# Show results
res_df = pd.DataFrame(results)
print("\nChosen strikes for", TARGET_DATE_STR)
print(res_df.to_string(index=False))



Chosen strikes for 20-10-2023
             Minute   ATM  CE_ATM_premium  PE_ATM_premium  CE_OTM_strike  CE_OTM_premium  PE_OTM_strike  PE_OTM_premium
2023-10-20 10:30:00 19550           97.75           96.25        19700.0           39.25        19400.0            42.5
2023-10-20 12:00:00 19550           88.35          106.55        19700.0           35.00        19400.0            47.5
2023-10-20 14:00:00 19550           85.65          102.30        19700.0           32.90        19400.0            44.1


In [8]:
# import pandas as pd

# def get_strike_debug(
#     ATM: int | float,
#     minute: pd.Timestamp,
#     daily_option_data: pd.DataFrame,
#     *,
#     strike_mult: float,               # replaces global STRIKE for clarity
#     price_col: str = "Open",
#     debug: bool = True
# ):
#     """
#     Debuggable version of get_strike:
#     - Verifies inputs
#     - Prints/logs intermediate values
#     - Lets you choose price column and multiplier explicitly
#     """
#     # --- input checks ---
#     required_cols = {"StrikePrice", "Type", price_col}
#     if not required_cols.issubset(set(daily_option_data.columns)):
#         raise ValueError(f"daily_option_data missing {required_cols - set(daily_option_data.columns)}")
#     if not isinstance(daily_option_data.index, pd.DatetimeIndex):
#         raise TypeError("daily_option_data.index must be a DatetimeIndex")
#     if minute.tzinfo and (daily_option_data.index.tz is None):
#         # optional: align TZ if needed
#         minute = minute.tz_convert(None)

#     # minute filter
#     time_str = minute.strftime("%H:%M:%S")
#     filter_time_object = pd.to_datetime(time_str).time()
#     temp_option_data = daily_option_data[daily_option_data.index.time == filter_time_object]
#     if debug:
#         print(f"\n=== {minute.date()} {time_str} === rows@minute: {len(temp_option_data)}  (ATM={ATM})")

#     if temp_option_data.empty:
#         if debug:
#             print("No rows at this minute.")
#         return (None, None, None, None, None, None)

#     # helper: ATM premium
#     def get_premium(option_type: str):
#         subset = temp_option_data[(temp_option_data["StrikePrice"] == ATM) & (temp_option_data["Type"] == option_type)]
#         if subset.empty:
#             if debug:
#                 print(f"ATM {option_type} not found at {time_str}")
#             return None
#         prem = subset[price_col].iloc[0]
#         if debug:
#             print(f"ATM {option_type} premium ({price_col}): {prem}")
#         return prem

#     # helper: nearest by premium (within the minute)
#     def find_nearest_strike(target_premium: float, option_type: str):
#         subset = temp_option_data[temp_option_data["Type"] == option_type]
#         if subset.empty:
#             if debug:
#                 print(f"No {option_type} rows at {time_str}")
#             return None, None
#         diffs = (subset[price_col] - target_premium).abs()
#         nearest_idx = diffs.idxmin()
#         row = subset.loc[nearest_idx]
#         return row["StrikePrice"], row[price_col]

#     # ATM premiums
#     ce_atm = get_premium("CE")
#     pe_atm = get_premium("PE")
#     if (ce_atm is None) or (pe_atm is None):
#         if debug:
#             print("Missing ATM CE/PE -> returning Nones.")
#         return (ce_atm, pe_atm, None, None, None, None)

#     # targets
#     ce_target = ce_atm * strike_mult
#     pe_target = pe_atm * strike_mult
#     if debug:
#         print(f"Target CE premium: {ce_target:.4f}  |  Target PE premium: {pe_target:.4f}")

#     # nearest OTM
#     ce_strike, ce_prem = find_nearest_strike(ce_target, "CE")
#     pe_strike, pe_prem = find_nearest_strike(pe_target, "PE")

#     if debug:
#         print(f"Chosen CE: strike={ce_strike}, premium={ce_prem} | "
#               f"Chosen PE: strike={pe_strike}, premium={pe_prem}")

#     return (ce_atm, pe_atm, ce_strike, ce_prem, pe_strike, pe_prem)


# def debug_strike_choices_for_day(
#     df_day: pd.DataFrame,
#     atm_by_minute: dict,
#     *,
#     strike_mult: float,
#     price_col: str = "Open",
#     minutes: list[pd.Timestamp] | None = None,
#     save_csv_path: str | None = None,
#     verbose: bool = True,
# ) -> pd.DataFrame:
#     """
#     df_day: one-day options DataFrame with DatetimeIndex
#     atm_by_minute: dict mapping minute -> ATM (lets you pass the ATM you want at each minute)
#                    e.g., {pd.Timestamp('2025-09-16 10:30'): 25000, ...}
#     minutes: optional list of timestamps to test; if None, infer unique minutes from df_day
#     Returns a DataFrame with chosen strikes and premiums per minute.
#     """
#     if minutes is None:
#         # infer unique HH:MM:SS buckets present in data
#         minutes = sorted(df_day.index.floor("min").unique())

#     rows = []
#     for m in minutes:
#         ATM = atm_by_minute.get(m)
#         if ATM is None:
#             if verbose:
#                 print(f"[skip] No ATM provided for {m}")
#             continue

#         ce_atm, pe_atm, ce_strike, ce_prem, pe_strike, pe_prem = get_strike_debug(
#             ATM=ATM,
#             minute=m,
#             daily_option_data=df_day,
#             strike_mult=strike_mult,
#             price_col=price_col,
#             debug=verbose
#         )

#         rows.append({
#             "Minute": m,
#             "ATM": ATM,
#             "CE_ATM_premium": ce_atm,
#             "PE_ATM_premium": pe_atm,
#             "CE_OTM_strike": ce_strike,
#             "CE_OTM_premium": ce_prem,
#             "PE_OTM_strike": pe_strike,
#             "PE_OTM_premium": pe_prem
#         })

#     out = pd.DataFrame(rows).sort_values("Minute")
#     if save_csv_path:
#         out.to_csv(save_csv_path, index=False)
#     return out


In [9]:
import pandas as pd

# 1) Ensure a datetime column exists
df_all['Datetime'] = pd.to_datetime(df_all['Datetime'], errors='coerce')

# 2) Drop bad rows and sort
df_all = df_all.dropna(subset=['Datetime']).sort_values('Datetime')

# 3) Set as index
df_all = df_all.set_index('Datetime')

# 4) (optional) Ensure tz-naive to avoid equality issues
if df_all.index.tz is not None:
    df_all.index = df_all.index.tz_convert(None)

# Now this works:
date_str = "2023-10-20"
df_day = df_all.loc[date_str]
df_day = df_day.sort_index()


KeyError: 'Datetime'

In [ ]:
# Suppose df_all is your full options dataset (many days).
# Filter to one day:
date_str = "2023-10-20"
df_day = df_all.loc[date_str]            # requires DatetimeIndex
df_day = df_day.sort_index()

# Pick the minutes you care about, and the ATM to check at those minutes:
mins = [
    pd.Timestamp(f"{date_str} 10:30:00"),
    pd.Timestamp(f"{date_str} 12:00:00"),
    pd.Timestamp(f"{date_str} 14:00:00"),
]
atm_map = {
    pd.Timestamp(f"{date_str} 10:30:00"): 25000,
    pd.Timestamp(f"{date_str} 12:00:00"): 25000,
    pd.Timestamp(f"{date_str} 14:00:00"): 25200,
}

res = debug_strike_choices_for_day(
    df_day=df_day,
    atm_by_minute=atm_map,
    strike_mult=0.40,          # your STRIKE multiplier
    price_col="Open",          # or "Close" / "LTP" if present
    minutes=mins,
    save_csv_path="chosen_strikes_2025-09-16.csv",
    verbose=True
)
print(res)


KeyError: '2023-10-20'